In [2]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer, ENGLISH_STOP_WORDS
from sklearn.cluster import KMeans
import re

In [3]:
# 1. Load the Data
file_path = 'Database_Task_1.xlsx'
xls = pd.ExcelFile(file_path)
df = pd.read_excel(xls, sheet_name=0)

In [4]:
# 2. Handle Translations (Merge Columns D and E)
# Assuming Column D is at index 3 and Column E is at index 4
col_d = df.columns[3]
col_e = df.columns[4]

# If a translation exists in Column E, use it; otherwise, use the original text in Column D
df['Unified_Text'] = df[col_e].combine_first(df[col_d])

In [5]:
# 3. Clean the Text
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'[^a-z\s]', '', text) # Remove punctuation and numbers
    return text

df['Cleaned_Text'] = df['Unified_Text'].apply(clean_text)

In [6]:
# 4. Feature Extraction (TF-IDF Vectorization)
tfidf = TfidfVectorizer(stop_words=list(ENGLISH_STOP_WORDS), max_features=500)
X = tfidf.fit_transform(df['Cleaned_Text'])

In [7]:
# 5. Clustering (K-Means)
num_clusters = 4
kmeans = KMeans(n_clusters=num_clusters, random_state=42, n_init=10)
df['Cluster'] = kmeans.fit_predict(X)

In [8]:
# 6. View the Top Terms per Cluster
order_centroids = kmeans.cluster_centers_.argsort()[:, ::-1]
terms = tfidf.get_feature_names_out()

for i in range(num_clusters):
    print(f"\n--- Cluster {i} Top Terms ---")
    top_terms = [terms[ind] for ind in order_centroids[i, :8]]
    print(", ".join(top_terms))



--- Cluster 0 Top Terms ---
leadership, icu, nursing, physician, unit, makes, clear, intervention

--- Cluster 1 Top Terms ---
nurses, interprofessional, groups, patient, work, professionals, group, physicians

--- Cluster 2 Top Terms ---
patients, care, facilitys, facility, clinicians, dying, health, information

--- Cluster 3 Top Terms ---
team, members, ip, care, patientclient, leader, patient, feel


In [9]:
# 7. Save the clustered data back to a new Excel file
df.to_excel('Database_Task_1_Clustered.xlsx', index=False)
print("\nClustering complete! File saved as 'Database_Task_1_Clustered.xlsx'")


Clustering complete! File saved as 'Database_Task_1_Clustered.xlsx'
